In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

NVIDIA GeForce RTX 2060
VRAM: 6.44 GB


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
import re

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

adapter_dir = f"../models/fine_tuned_model/train"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    offload_folder="offload",
)

model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


Loading checkpoint shards: 100%|██████████| 3/3 [00:37<00:00, 12.40s/it]


In [ ]:
def build_prompt(sample):
    system = "You are a helpful assistant that answers questions about food and health relationships based on scientific evidence. Always cite your sources with page numbers."
    
    instruction = sample
    user_message = f"{instruction}"

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_message},
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False)

In [ ]:
def generate_answer(sample, max_new_tokens=256):
    prompt = build_prompt(sample)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    decoded = decoded.split("assistant")[-1].strip()
    _, _, answer = decoded.partition("[/INST]")
    answer = answer.strip()
    
    pattern = r"\(Source:\s*(?:p\.?\s*\d+(?:,\s*p\.?\s*\d+)*)\)"

    # Extract all page numbers inside the parentheses
    pages = re.findall(r"p\.?\s*(\d+)", answer)

    # Remove the full (Source: ...) block
    cleaned_text = re.sub(pattern, "", answer).strip()

    return cleaned_text, pages